In [ ]:
import numpy as np
import pandas as pd
df = pd.read_csv("all_stock_ratios_df.csv")
display(df.head())

,fiscalDateEnding,Ticker,net_income_minus_taxes_over_net_income,pretax_non_operating_over_sales,sales_over_assets,income_to_equity,earnings_over_price,relative_price_evolution,net_income_to_common_over_net_income_smoothed,liabilities_over_equity,inventory_over_current_liabilities,working_capital_over_sales,dividend_per_share
0,2017-03-01,LECO,0.605114,-0.006173,0.282995,0.071218,0.193118,0.031469,0.0,1.617796,0.666054,1.249459,22.900740
1,2017-04-01,LECO,0.421442,-0.006505,0.301234,0.043143,0.114678,0.057238,0.0,1.537092,0.657183,1.243505,21.446891
2,2017-05-01,LECO,0.400729,-0.006357,0.303231,0.048669,0.132847,0.061394,0.0,1.498753,0.671808,1.244453,21.491741
3,2017-06-01,LECO,0.409378,-0.005864,0.295455,0.072028,0.199637,0.093576,0.0,1.490881,0.696836,1.249215,22.608214
4,2017-07-01,LECO,0.586231,-0.005177,0.285221,0.097321,0.297908,0.036219,0.0,1.502741,0.717163,1.262544,24.315648


In [ ]:
import numpy as np
import pandas as pd

ratios_to_remove = [
    'relative_price_evolution',
    'pretax_non_operating_over_sales'
]


ratio_columns = [col for col in df.columns
                 if col not in ['fiscalDateEnding', 'Ticker'] + ratios_to_remove]

print(f"Keeping {len(ratio_columns)} ratios: {ratio_columns}")


for col in ratio_columns:
    if df[col].dtype in ['float64', 'float32']:
        inf_count = np.isinf(df[col]).sum()
        if inf_count > 0:
            print(f"WARNING: {col} has {inf_count} infinite values")

Keeping 9 ratios: ['net_income_minus_taxes_over_net_income', 'sales_over_assets', 'income_to_equity', 'earnings_over_price', 'net_income_to_common_over_net_income_smoothed', 'liabilities_over_equity', 'inventory_over_current_liabilities', 'working_capital_over_sales', 'dividend_per_share']


In [ ]:
for col in ratio_columns:
    df[col] = df[col].replace([np.inf, -np.inf], np.nan)


df[ratio_columns] = df.groupby('Ticker')[ratio_columns].fillna(method='ffill')
df[ratio_columns] = df.groupby('Ticker')[ratio_columns].fillna(method='bfill')


for col in ratio_columns:
    df[col] = df[col].fillna(df[col].median())

/tmp/ipykernel_5659/191948611.py:5: FutureWarning: DataFrameGroupBy.fillna is deprecated and will be removed in a future version. Use obj.ffill() or obj.bfill() for forward or backward filling instead. If you want to fill with a single value, use DataFrame.fillna instead
  df[ratio_columns] = df.groupby('Ticker')[ratio_columns].fillna(method='ffill')
/tmp/ipykernel_5659/191948611.py:5: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df[ratio_columns] = df.groupby('Ticker')[ratio_columns].fillna(method='ffill')
/tmp/ipykernel_5659/191948611.py:6: FutureWarning: DataFrameGroupBy.fillna is deprecated and will be removed in a future version. Use obj.ffill() or obj.bfill() for forward or backward filling instead. If you want to fill with a single value, use DataFrame.fillna instead
  df[ratio_columns] = df.groupby('Ticker')[ratio_columns].fillna(method='bfill')
/tmp/ipykernel_5659/191948611.py:6: Futur

In [ ]:
from scipy.stats.mstats import winsorize

for col in ratio_columns:
    df[col] = winsorize(df[col], limits=(0.01, 0.01))

In [ ]:
num_unique_tickers = len(set(df["Ticker"]))
print(f"Number of unique stock tickers: {num_unique_tickers}")

Number of unique stock tickers: 182


In [ ]:
import numpy as np

ratios_to_remove = [
    'relative_price_evolution',
    'pretax_non_operating_over_sales'
]


ratio_columns = [col for col in df.columns
                 if col not in ['fiscalDateEnding', 'Ticker'] + ratios_to_remove]

print(f"Updated ratio_columns to: {ratio_columns}")

Updated ratio_columns to: ['net_income_minus_taxes_over_net_income', 'sales_over_assets', 'income_to_equity', 'earnings_over_price', 'net_income_to_common_over_net_income_smoothed', 'liabilities_over_equity', 'inventory_over_current_liabilities', 'working_capital_over_sales', 'dividend_per_share']


In [ ]:
X_numpy = []
tickers = []

for ticker, group in df.groupby('Ticker'):
    group_sorted = group.sort_values('fiscalDateEnding')
    ratio_values = group_sorted[ratio_columns].to_numpy()

    # Skip stocks with too few periods
    if len(ratio_values) < 12:  # Need at least 12 periods
        continue

    X_numpy.append(ratio_values)
    tickers.append(ticker)

# Find the most common shape among the collected arrays
if X_numpy:
    from collections import Counter
    shapes = [arr.shape for arr in X_numpy]
    shape_counts = Counter(shapes)
    most_common_shape = shape_counts.most_common(1)[0][0]


    X_numpy_filtered = []
    tickers_filtered = []
    for i, arr in enumerate(X_numpy):
        if arr.shape == most_common_shape:
            X_numpy_filtered.append(arr)
            tickers_filtered.append(tickers[i])

    X_multivariate = np.stack(X_numpy_filtered)
    tickers = tickers_filtered
else:
    X_multivariate = np.array([])
    tickers = []

print(f"Shape: {X_multivariate.shape}")

Shape: (172, 58, 9)


In [ ]:
from sklearn.preprocessing import StandardScaler

X_multivariate[np.isinf(X_multivariate)] = np.nan


if np.isnan(X_multivariate).any():
    print("Warning: NaN values found in X_multivariate before scaling. Filling with 0.0.")
    X_multivariate = np.nan_to_num(X_multivariate, nan=0.0)

print(f"Number of infinite values in X_multivariate after cleaning: {np.isinf(X_multivariate).sum()}")
print(f"Number of NaN values in X_multivariate after cleaning: {np.isnan(X_multivariate).sum()}")

n_stocks, n_periods, n_ratios = X_multivariate.shape
X_2d = X_multivariate.reshape(-1, n_ratios)
scaler = StandardScaler()
X_2d_scaled = scaler.fit_transform(X_2d)
X_scaled = X_2d_scaled.reshape(n_stocks, n_periods, n_ratios)

# Check for outliers again
stock_stds = X_scaled.std(axis=(1, 2))
print(f"Std dev range: {stock_stds.min():.2f} to {stock_stds.max():.2f}")
print(f"Stocks with std > 2.5: {(stock_stds > 2.5).sum()}")

Number of infinite values in X_multivariate after cleaning: 0
Number of NaN values in X_multivariate after cleaning: 0
Std dev range: 0.35 to 3.28
Stocks with std > 2.5: 2


In [ ]:

X_multivariate[np.isinf(X_multivariate)] = np.nan


if np.isnan(X_multivariate).any():
    print("Warning: NaN values found in X_multivariate. Filling with 0.")
    X_multivariate = np.nan_to_num(X_multivariate, nan=0.0)

print(f"Number of infinite values in X_multivariate after cleaning: {np.isinf(X_multivariate).sum()}")
print(f"Number of NaN values in X_multivariate after cleaning: {np.isnan(X_multivariate).sum()}")

Number of infinite values in X_multivariate after cleaning: 0
Number of NaN values in X_multivariate after cleaning: 0


In [ ]:
print(f"\nShape: {X_multivariate.shape}")
print(f"  - Stocks: {X_multivariate.shape[0]}")
print(f"  - Time periods (quarters): {X_multivariate.shape[1]}")
print(f"  - Ratios: {X_multivariate.shape[2]}")


Shape: (172, 58, 9)
  - Stocks: 172
  - Time periods (quarters): 58
  - Ratios: 9


In [ ]:
nan_count = np.isnan(X_multivariate).sum()
print(f"\nNaN values: {nan_count}")


NaN values: 0


In [ ]:
volatile_stocks = ['INSM', 'LBRDA', 'VHC','CMCT']


keep_mask = [t not in volatile_stocks for t in tickers]


X_clean = X_scaled[keep_mask]


tickers_clean = [t for t in tickers if t not in volatile_stocks]

print(f"Removed {len(tickers) - len(tickers_clean)} stocks due to volatility.")
print(f"New shape: {X_clean.shape}")
print(f"New number of tickers: {len(tickers_clean)}")

Removed 4 stocks due to volatility.
New shape: (168, 58, 9)
New number of tickers: 168


In [ ]:
!pip install tslearn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 387.9/387.9 kB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.8/3.8 MB 76.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 19.0 MB/s eta 0:00:00
  Attempting uninstall: llvmlite
    Found existing installation: llvmlite 0.43.0
    Uninstalling llvmlite-0.43.0:
      Successfully uninstalled llvmlite-0.43.0
  Attempting uninstall: numba
    Found existing installation: numba 0.60.0
    Uninstalling numba-0.60.0:
      Successfully uninstalled numba-0.60.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.0 which is incompatible.
cudf-cu12 26.2.1 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.0 which is incompatible.


In [ ]:
from tslearn.clustering import TimeSeriesKMeans
from tslearn.metrics import cdist_dtw
from sklearn.metrics import silhouette_score


keep_mask = stock_stds < 2.5
X_clean = X_scaled[keep_mask]
tickers_clean = [tickers[i] for i in range(len(tickers)) if keep_mask[i]]

print(f"Stocks after outlier removal: {len(X_clean)}")


best_k = 2
best_score = -1

for k in range(2, 16):
    model = TimeSeriesKMeans(n_clusters=k, metric="dtw", max_iter=10, random_state=42, n_init=10)
    labels = model.fit_predict(X_clean)

    dist_matrix = cdist_dtw(X_clean)
    score = silhouette_score(dist_matrix, labels, metric='precomputed')

    unique, counts = np.unique(labels, return_counts=True)
    print(f"k={k}: silhouette={score:.4f}, sizes={dict(zip(unique, counts))}")

    if score > best_score:
        best_score = score
        best_k = k

print(f"\n Best k={best_k}, silhouette={best_score:.4f}")

Stocks after outlier removal: 170
k=2: silhouette=0.1180, sizes={np.int64(0): np.int64(88), np.int64(1): np.int64(82)}
k=3: silhouette=0.1291, sizes={np.int64(0): np.int64(73), np.int64(1): np.int64(46), np.int64(2): np.int64(51)}
k=4: silhouette=0.1747, sizes={np.int64(0): np.int64(24), np.int64(1): np.int64(20), np.int64(2): np.int64(95), np.int64(3): np.int64(31)}
k=5: silhouette=0.1606, sizes={np.int64(0): np.int64(77), np.int64(1): np.int64(40), np.int64(2): np.int64(23), np.int64(3): np.int64(27), np.int64(4): np.int64(3)}
k=6: silhouette=0.1449, sizes={np.int64(0): np.int64(20), np.int64(1): np.int64(62), np.int64(2): np.int64(6), np.int64(3): np.int64(35), np.int64(4): np.int64(4), np.int64(5): np.int64(43)}
k=7: silhouette=0.1725, sizes={np.int64(0): np.int64(9), np.int64(1): np.int64(81), np.int64(2): np.int64(30), np.int64(3): np.int64(9), np.int64(4): np.int64(30), np.int64(5): np.int64(9), np.int64(6): np.int64(2)}
k=8: silhouette=0.1505, sizes={np.int64(0): np.int64(34), 